In [ ]:
# First-round sampling and reuse of saved splits.
# Run from the project root or first_round_data/.
# Saved cell outputs are historical and predate the directory move.

In [18]:
# Load source data and validate inputs.
import hashlib
import json
import random
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
if ((cwd / "select_data_first_round.ipynb").is_file()):
    data_dir = cwd
elif ((cwd / "first_round_data" / "select_data_first_round.ipynb").is_file()):
    data_dir = cwd / "first_round_data"
else:
    raise ValueError("Run this notebook from the project root or first_round_data/.")


def data_path(value):
    path = Path(value).expanduser()
    path = (path if path.is_absolute() else data_dir / path).resolve()
    if (path != data_dir and data_dir not in path.parents):
        raise ValueError(f"Paths must stay inside {data_dir}: {path}")
    return path


source_path = data_path("raw/evasionbench_17k_eva4b_labels_dedup.parquet")
GOLD_SEED = 42
LLM_SEED = 43
REQUESTED_MODEL = "gpt-6-astra"
SOURCE_SHA256 = "8ca4194c00cb037e9d719d45a569935aaabb31c3f781f03c1fcb9c7db875508b"
source_sha256 = hashlib.sha256(source_path.read_bytes()).hexdigest()
if source_sha256 != SOURCE_SHA256:
    raise ValueError("The source dataset differs from the frozen sampling reference.")
records = pd.read_parquet(source_path, columns=["uid", "question", "answer", "eva4b_label"]).to_dict("records")
labels = ['direct', 'intermediate', 'fully_evasive']
by_uid = {row["uid"]: i for i, row in enumerate(records)}
if len(by_uid) != len(records):
    raise ValueError("Duplicate source UIDs.")
if any(not isinstance(row[key], str) or not row[key].strip()
       for row in records for key in ("uid", "question", "answer")):
    raise ValueError("Missing source UID, question, or answer.")
if any(row["eva4b_label"] not in labels for row in records):
    raise ValueError("Unexpected original label.")


In [19]:
# Group duplicate samples and define sampling and export functions.
def normalize(text):
    return " ".join(unicodedata.normalize("NFKC", text).casefold().split())


parent = list(range(len(records)))


def find_group(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


seen = {}
for i, row in enumerate(records):
    keys = [("question", normalize(row["question"]))]
    answer = normalize(row["answer"])
    if len(answer.split()) >= 20:
        keys.append(("answer", answer))
    for key in keys:
        if key in seen:
            parent[find_group(i)] = find_group(seen[key])
        else:
            seen[key] = i
groups = [find_group(i) for i in range(len(records))]


def select_splits(blocked_uids, sizes, seed, proportional=False, required_uids=()):
    if not sizes or any(type(size) is not int or size <= 0 for size in sizes.values()):
        raise ValueError("Each split size must be a positive integer.")
    if not proportional and any(size % len(labels) for size in sizes.values()):
        raise ValueError("Balanced split sizes must be divisible by three.")
    blocked_groups = {groups[by_uid[uid]] for uid in blocked_uids}
    required_uids = list(required_uids)
    if required_uids and len(sizes) != 1:
        raise ValueError("Required examples can only be assigned to one split at a time.")
    if len(set(required_uids)) != len(required_uids) or any(uid not in by_uid for uid in required_uids):
        raise ValueError("Required examples must have unique, known UIDs.")
    required_indices = [by_uid[uid] for uid in required_uids]
    required_groups = {groups[i] for i in required_indices}
    if len(required_groups) != len(required_indices):
        raise ValueError("Required examples contain duplicate sample groups.")
    if required_groups & blocked_groups:
        raise ValueError("Required examples overlap an excluded sample group.")
    required_set = set(required_indices)
    required_counts = Counter(records[i]["eva4b_label"] for i in required_indices)
    eligible = [i for i in range(len(records)) if groups[i] not in blocked_groups]
    rng = random.Random(seed)
    rng.shuffle(eligible)
    candidates, selected_groups = list(required_indices), set(required_groups)
    for i in eligible:
        if groups[i] not in selected_groups:
            candidates.append(i)
            selected_groups.add(groups[i])
    pools = {label: [i for i in candidates if records[i]["eva4b_label"] == label] for label in labels}
    candidate_counts = {label: len(pool) for label, pool in pools.items()}
    total = len(candidates)
    if sum(sizes.values()) > total:
        raise ValueError(f"Only {total} independent candidates remain.")
    quotas = {}
    for name, size in sizes.items():
        if proportional:
            products = {label: size * candidate_counts[label] for label in labels}
            counts = {label: products[label] // total for label in labels}
            remainder_order = sorted(labels, key=lambda label: -(products[label] % total))
            for label in remainder_order[:size - sum(counts.values())]:
                counts[label] += 1
            quotas[name] = counts
        else:
            quotas[name] = {label: size // len(labels) for label in labels}
    splits = {name: list(required_indices) for name in sizes}
    for label, pool in pools.items():
        available = [i for i in pool if i not in required_set]
        needed = {name: quotas[name][label] - required_counts[label] for name in sizes}
        if any(count < 0 for count in needed.values()):
            raise ValueError(f"Required {label} examples exceed the proportional quota; increase the split size.")
        required = sum(needed.values())
        if len(available) < required:
            raise ValueError(f"Only {len(available)} independent {label} candidates remain; {required} required.")
        offset = 0
        for name in sizes:
            count = needed[name]
            splits[name].extend(available[offset:offset + count])
            offset += count
    for selected in splits.values():
        rng.shuffle(selected)
    return splits, candidate_counts


def reserved_uids(indices):
    selected_groups = {groups[i] for i in indices}
    return [row["uid"] for i, row in enumerate(records) if groups[i] in selected_groups]


def human_template(i):
    row = records[i]
    return {
        "uid": row["uid"], "source_row": i + 1,
        "question": row["question"], "answer": row["answer"],
        "overall_label": None, "overall_rationale": "", "reviewer": "",
        "annotation_status": "draft", "notes": "",
    }


def joint_template(i):
    row = human_template(i)
    row.update(
        num_sub_questions=None,
        sub_questions=[""],
        sub_answers=[None],
        decomposition_issue=None,
        annotations=[{"label": None, "rationale": "", "evidence": [], "needs_review": True}],
    )
    return row


def folder_has_content(directory):
    directory = data_path(directory)
    if directory.exists() and not directory.is_dir():
        raise NotADirectoryError(directory)
    return directory.is_dir() and next(directory.iterdir(), None) is not None


def load_saved_split(directory, file_sizes):
    directory = data_path(directory)
    missing = [name for name in file_sizes if not data_path(directory / name).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Required JSONL files are missing in {directory}: {missing}. "
            "Restore the existing split files before continuing; nothing was overwritten."
        )
    rows_by_file, seen_groups = {}, set()
    for name, size in file_sizes.items():
        rows = [json.loads(line) for line in data_path(directory / name).read_text(encoding="utf-8").splitlines()
                if line.strip()]
        if len(rows) != size:
            raise ValueError(f"Expected {size} saved rows in {directory / name}; found {len(rows)}.")
        for row in rows:
            if not isinstance(row, dict) or row.get("uid") not in by_uid:
                raise ValueError(f"Unknown saved UID in {directory / name}.")
            i = by_uid[row["uid"]]
            if any(row.get(key) != records[i][key] for key in ("question", "answer")):
                raise ValueError(f"Saved question or answer changed for {row['uid']}.")
            if groups[i] in seen_groups:
                raise ValueError(f"Repeated sample group in {directory}: {row['uid']}.")
            seen_groups.add(groups[i])
        rows_by_file[name] = rows
    return rows_by_file


def save_split_files(directory, rows_by_file):
    directory = data_path(directory)
    payloads = {name: "".join(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n" for row in rows).encode("utf-8")
                for name, rows in rows_by_file.items()}
    for name, raw in payloads.items():
        path = data_path(directory / name)
        if path.exists() and path.read_bytes() != raw:
            raise FileExistsError(f"Existing content differs; not overwriting {path}.")
    directory.mkdir(parents=True, exist_ok=True)
    for name, raw in payloads.items():
        path = data_path(directory / name)
        if not path.exists():
            with path.open("xb") as stream:
                stream.write(raw)
    return directory


In [20]:
# Reuse the six human annotation batches, or select 120 balanced samples.
gold_output = data_path("human_annotated_data/first_round")
gold_exists = folder_has_content(gold_output)
if gold_exists:
    print(f"Skipped generation: {gold_output} is not empty. Loading the saved Gold samples.")
    gold_batches = load_saved_split(
        gold_output, {f"batch_{number:02d}.jsonl": 20 for number in range(1, 7)}
    )
    gold_templates = [row for rows in gold_batches.values() for row in rows]
    gold_uids = [row["uid"] for row in gold_templates]
    gold_indices = [by_uid[uid] for uid in gold_uids]
else:
    gold_splits, _ = select_splits(set(), {"gold": 120}, seed=GOLD_SEED)
    gold_indices = gold_splits["gold"]
    gold_uids = [records[i]["uid"] for i in gold_indices]
    gold_templates = [human_template(i) for i in gold_indices]
    gold_batches = {f"batch_{start // 20 + 1:02d}.jsonl": gold_templates[start:start + 20]
                    for start in range(0, 120, 20)}
gold_reserved = reserved_uids(gold_indices)
if not gold_exists:
    save_split_files(gold_output, gold_batches)
print(gold_output)
pd.DataFrame([
    {"batch": name, "count": len(rows),
     **dict(Counter(records[by_uid[row["uid"]]]["eva4b_label"] for row in rows))}
    for name, rows in gold_batches.items()
])


Skipped generation: /Users/zyliu/Documents/ChatGPT/NLP_project/Detecting_Incomplete_and_Evasive_Answers/code/data/human_annotations/gold_test_120total is not empty. Loading the saved Gold samples.
/Users/zyliu/Documents/ChatGPT/NLP_project/Detecting_Incomplete_and_Evasive_Answers/code/data/human_annotations/gold_test_120total


,batch,count,intermediate,direct,fully_evasive
0,batch_01.jsonl,20,6,7,7
1,batch_02.jsonl,20,7,8,5
2,batch_03.jsonl,20,7,7,6
3,batch_04.jsonl,20,6,8,6
4,batch_05.jsonl,20,8,3,9
5,batch_06.jsonl,20,6,7,7


In [21]:
# Reuse existing LLM splits, or select proportional 300-sample dev and test sets.
llm_output = data_path("LLM_annotations/llm_dev_test_300")
llm_exists = folder_has_content(llm_output)
if llm_exists:
    print(f"Skipped generation: {llm_output} is not empty. Loading the saved dev/test samples.")
    llm_rows = load_saved_split(llm_output, {"dev_300.jsonl": 300, "test_300.jsonl": 300})
    llm_uids = {split: [row["uid"] for row in llm_rows[f"{split}_300.jsonl"]] for split in ("dev", "test")}
    llm_splits = {split: [by_uid[uid] for uid in uids] for split, uids in llm_uids.items()}
else:
    llm_splits, _ = select_splits(
        set(gold_reserved), {"dev": 300, "test": 300},
        seed=LLM_SEED, proportional=True,
    )
    llm_uids = {split: [records[i]["uid"] for i in indices] for split, indices in llm_splits.items()}
    llm_rows = {}
    for split, indices in llm_splits.items():
        rows = []
        for i in indices:
            row = joint_template(i)
            row.update(split=split, requested_model=REQUESTED_MODEL,
                       llm_label=None, llm_rationale="", annotation_status="pending_llm")
            rows.append(row)
        llm_rows[f"{split}_300.jsonl"] = rows
heldout_groups = {"gold": {groups[i] for i in gold_indices},
                  **{split: {groups[i] for i in indices} for split, indices in llm_splits.items()}}
for split, split_groups in heldout_groups.items():
    for other, other_groups in heldout_groups.items():
        if split != other and split_groups & other_groups:
            raise ValueError(f"Group overlap between {split} and {other}.")
reservations = {"gold": gold_reserved,
                **{split: reserved_uids(indices) for split, indices in llm_splits.items()}}
llm_label_counts = {
    split: {label: sum(records[i]["eva4b_label"] == label for i in indices) for label in labels}
    for split, indices in llm_splits.items()
}
if not llm_exists:
    save_split_files(llm_output, llm_rows)
print(llm_output)
pd.DataFrame([
    {"original_label": label, **{f"{split}_count": counts[label] for split, counts in llm_label_counts.items()}}
    for label in labels
])


Skipped generation: /Users/zyliu/Documents/ChatGPT/NLP_project/Detecting_Incomplete_and_Evasive_Answers/code/data/LLM_annotations/llm_dev_test_300 is not empty. Loading the saved dev/test samples.
/Users/zyliu/Documents/ChatGPT/NLP_project/Detecting_Incomplete_and_Evasive_Answers/code/data/LLM_annotations/llm_dev_test_300


,original_label,dev_count,test_count
0,direct,157,157
1,intermediate,134,134
2,fully_evasive,9,9


In [22]:
# Reuse existing team files, or select balanced examples excluding all held-out groups.
N_EXAMPLES = 12
REVIEWERS = []
EXAMPLE_SEED = 44


def export_team_examples(size, reviewers=(), seed=44):
    if type(size) is not int or size <= 0 or size % len(labels):
        raise ValueError("N_EXAMPLES must be a positive multiple of three.")
    output = data_path(f"human_annotated_data/gold_standard_{size}example")
    existing = folder_has_content(output)
    if existing:
        print(f"Skipped generation: {output} is not empty. Loading the saved team samples.")
        filename = "consensus.jsonl" if data_path(output / "consensus.jsonl").is_file() else "annotation_template.jsonl"
        saved = load_saved_split(output, {filename: size})
        uids = [row["uid"] for row in saved[filename]]
        if filename == "consensus.jsonl" and data_path(output / "annotation_template.jsonl").is_file():
            templates = load_saved_split(output, {"annotation_template.jsonl": size})
            if uids != [row["uid"] for row in templates["annotation_template.jsonl"]]:
                raise ValueError("Consensus and annotation template UIDs differ.")
        indices = [by_uid[uid] for uid in uids]
    else:
        if not isinstance(reviewers, (list, tuple)) or any(
            not isinstance(name, str) or not name.strip() for name in reviewers
        ):
            raise ValueError("REVIEWERS must contain nonempty reviewer names.")
        reviewers = [name.strip() for name in reviewers]
        if len(set(reviewers)) != len(reviewers):
            raise ValueError("Reviewer names must be unique.")
    llm_rows = load_saved_split(
        data_dir / "LLM_annotations/llm_dev_test_300", {"dev_300.jsonl": 300, "test_300.jsonl": 300}
    )
    gold_rows = load_saved_split(
        data_path("human_annotated_data/first_round"),
        {f"batch_{number:02d}.jsonl": 20 for number in range(1, 7)},
    )
    heldout_uids = {"gold": [row["uid"] for rows in gold_rows.values() for row in rows],
                   **{split: [row["uid"] for row in llm_rows[f"{split}_300.jsonl"]] for split in ("dev", "test")}}
    if len(heldout_uids["gold"]) != 120 or len(set(heldout_uids["gold"])) != 120:
        raise ValueError("The saved Gold batches must contain 120 unique samples.")
    heldout_reservations = {
        split: reserved_uids([by_uid[uid] for uid in uids]) for split, uids in heldout_uids.items()
    }
    for split, uids in heldout_reservations.items():
        for other, other_uids in heldout_reservations.items():
            if split != other and set(uids) & set(other_uids):
                raise ValueError(f"Group overlap between {split} and {other}.")
    blocked = set().union(*(set(uids) for uids in heldout_reservations.values()))
    if not existing:
        selected, _ = select_splits(blocked, {"team": size}, seed=seed)
        indices = selected["team"]
    if set(reserved_uids(indices)) & blocked:
        raise ValueError("Team examples overlap a held-out group.")
    counts = dict(Counter(records[i]["eva4b_label"] for i in indices))
    if existing:
        return output, counts
    rows = [joint_template(i) for i in indices]
    files = {"annotation_template.jsonl": rows, "consensus.jsonl": rows}
    for number, reviewer in enumerate(reviewers, 1):
        files[f"reviewer_{number:02d}.jsonl"] = [{**row, "reviewer": reviewer} for row in rows]
    save_split_files(output, files)
    return output, counts


team_output, team_label_counts = export_team_examples(N_EXAMPLES, REVIEWERS, EXAMPLE_SEED)
print(team_output)
pd.DataFrame([{"original_label": label, "count": team_label_counts.get(label, 0)} for label in labels])


Skipped generation: /Users/zyliu/Documents/ChatGPT/NLP_project/Detecting_Incomplete_and_Evasive_Answers/code/data/human_annotations/gold_standard_12example is not empty. Loading the saved team samples.
/Users/zyliu/Documents/ChatGPT/NLP_project/Detecting_Incomplete_and_Evasive_Answers/code/data/human_annotations/gold_standard_12example


,original_label,count
0,direct,4
1,intermediate,4
2,fully_evasive,4


In [23]:
# Select 2,700 proportional training samples including all 12 Gold examples.
N_TRAIN = 2700
TRAIN_SEED = 45


def export_training_set(size=2700, seed=45):
    if type(size) is not int or size <= 12:
        raise ValueError("The training size must be an integer greater than 12.")
    output = data_path(f"LLM_annotations/llm_train_{size}")
    train_filename = f"train_{size}.jsonl"
    llm_filename = f"llm_input_{size - 12}.jsonl"
    existing = folder_has_content(output)
    human_rows = load_saved_split(
        data_path("human_annotated_data/first_round"),
        {f"batch_{number:02d}.jsonl": 20 for number in range(1, 7)},
    )
    llm_rows = load_saved_split(
        data_path("LLM_annotations/llm_dev_test_300"),
        {"dev_300.jsonl": 300, "test_300.jsonl": 300},
    )
    heldout_rows = [row for batches in (human_rows, llm_rows)
                    for rows in batches.values() for row in rows]
    heldout_indices = [by_uid[row["uid"]] for row in heldout_rows]
    if len({groups[i] for i in heldout_indices}) != 720:
        raise ValueError("The 120/300/300 held-out sets must contain 720 distinct sample groups.")
    blocked = set(reserved_uids(heldout_indices))
    example_dir = data_path("human_annotated_data/gold_standard_12example")
    example_rows = load_saved_split(example_dir, {"consensus.jsonl": 12})["consensus.jsonl"]
    required_uids = [row["uid"] for row in example_rows]
    template_path = data_path(example_dir / "annotation_template.jsonl")
    if template_path.is_file():
        templates = load_saved_split(example_dir, {template_path.name: 12})[template_path.name]
        if [row["uid"] for row in templates] != required_uids:
            raise ValueError("Consensus and annotation template UIDs differ.")
    if set(reserved_uids([by_uid[uid] for uid in required_uids])) & blocked:
        raise ValueError("The required 12 Gold examples overlap excluded sample groups.")
    examples_by_uid = {row["uid"]: row for row in example_rows}
    if existing:
        print(f"Skipped generation: {output} is not empty. Loading the saved training samples.")
        saved = load_saved_split(output, {train_filename: size})
        train_rows = saved[train_filename]
        indices = [by_uid[row["uid"]] for row in train_rows]
    else:
        selected, _ = select_splits(
            blocked, {"train": size}, seed=seed, proportional=True,
            required_uids=required_uids,
        )
        indices = selected["train"]
        train_rows = []
        for i in indices:
            uid = records[i]["uid"]
            if uid in examples_by_uid:
                row = dict(examples_by_uid[uid])
                row.update(split="train", is_gold_example=True,
                           gold_example_source=str(data_path(example_dir / "consensus.jsonl").relative_to(data_dir)))
            else:
                row = joint_template(i)
                row.update(split="train", is_gold_example=False, requested_model=REQUESTED_MODEL,
                           llm_label=None, llm_rationale="", annotation_status="pending_llm")
            train_rows.append(row)
    train_uids = {row["uid"] for row in train_rows}
    if not set(required_uids) <= train_uids:
        raise ValueError("The saved training set must include all 12 current Gold examples.")
    if len(indices) != size or len(train_uids) != size or len({groups[i] for i in indices}) != size:
        raise ValueError("Training samples must have the requested size and unique sample groups.")
    if set(reserved_uids(indices)) & blocked:
        raise ValueError("Training samples overlap held-out groups.")
    pending_rows = [row for row in train_rows if row["uid"] not in examples_by_uid]
    if existing:
        saved_inputs = load_saved_split(output, {llm_filename: size - 12})[llm_filename]
        if [row["uid"] for row in saved_inputs] != [row["uid"] for row in pending_rows]:
            raise ValueError("The saved LLM input must match the non-Gold training samples in order.")
    else:
        save_split_files(output, {train_filename: train_rows, llm_filename: pending_rows})
    counts = Counter(records[i]["eva4b_label"] for i in indices)
    example_counts = Counter(records[by_uid[uid]]["eva4b_label"] for uid in required_uids)
    summary = pd.DataFrame([
        {"original_label": label, "train_count": counts[label],
         "train_percent": round(100 * counts[label] / size, 2),
         "gold_examples": example_counts[label], "llm_pending_count": counts[label] - example_counts[label]}
        for label in labels
    ])
    return output, summary


train_output, train_label_counts = export_training_set(N_TRAIN, TRAIN_SEED)
print(train_output)
train_label_counts


Skipped generation: /Users/zyliu/Documents/ChatGPT/NLP_project/Detecting_Incomplete_and_Evasive_Answers/code/data/LLM_annotations/llm_train_2700 is not empty. Loading the saved training samples.
/Users/zyliu/Documents/ChatGPT/NLP_project/Detecting_Incomplete_and_Evasive_Answers/code/data/LLM_annotations/llm_train_2700


,original_label,train_count,train_percent,gold_examples,llm_pending_count
0,direct,1412,52.30,4,1408
1,intermediate,1205,44.63,4,1201
2,fully_evasive,83,3.07,4,79
